In [ ]:
%reload_ext autoreload
%autoreload 2

In [1]:
from sentence_transformers import SentenceTransformer

import sys
sys.path.append("../model_exploration/")  # Add training dir to path

from custum_evals import (
    UBinarySentenceTransformer
)



# Load your trained model from checkpoint
checkpoint = "/rhome/sawale/indus_traning/sentense_transformers/eval/artifacts/s2_binarized_model/azure-eon-73/checkpoint-32500"
model = SentenceTransformer(checkpoint)
model_binarized = UBinarySentenceTransformer(checkpoint)

No sentence-transformers model found with name /rhome/sawale/indus_traning/sentense_transformers/eval/artifacts/s2_binarized_model/azure-eon-73/checkpoint-32500. Creating a new one with mean pooling.


In [2]:
x = model.encode(["abc"])
y = model_binarized.encode(["abc"])

In [3]:
x

array([[ 1.,  1., -1.,  1.,  1., -1., -1.,  1.,  1., -1.,  1.,  1., -1.,
        -1., -1.,  1., -1., -1., -1.,  1., -1., -1., -1.,  1., -1.,  1.,
        -1., -1.,  1., -1.,  1.,  1., -1., -1.,  1.,  1.,  1.,  1.,  1.,
         1., -1.,  1., -1., -1.,  1., -1., -1.,  1., -1.,  1., -1., -1.,
        -1., -1., -1.,  1.,  1.,  1., -1., -1.,  1., -1.,  1.,  1.,  1.,
         1., -1.,  1., -1.,  1., -1.,  1., -1., -1.,  1., -1., -1.,  1.,
        -1.,  1., -1.,  1.,  1.,  1., -1.,  1., -1.,  1.,  1., -1., -1.,
         1., -1., -1., -1.,  1.,  1., -1.,  1., -1., -1.,  1., -1.,  1.,
         1., -1., -1., -1., -1., -1.,  1., -1.,  1., -1., -1.,  1.,  1.,
         1., -1.,  1., -1.,  1.,  1., -1.,  1.,  1., -1.,  1., -1., -1.,
         1., -1., -1.,  1.,  1.,  1.,  1., -1.,  1.,  1., -1., -1., -1.,
         1., -1.,  1.,  1., -1., -1., -1.,  1., -1.,  1., -1.,  1., -1.,
         1.,  1.,  1.,  1.,  1., -1.,  1., -1.,  1.,  1.,  1., -1.,  1.,
        -1.,  1.,  1.,  1.,  1., -1., -1.,  1.,  1.

In [4]:
y

tensor([[217, 177,  17,  75,  63,  73,  65, 203, 213,  37, 117, 145, 165, 130,
         157, 109,  39, 177,  98, 175, 174, 188, 216, 197,  90,  64, 229, 114,
          84,  62,  97,  11, 235,  83,  85, 130, 127, 129, 109, 238, 117,  76,
          89, 120,  81,  48, 244,  71,  72, 190, 121, 212,   3, 235, 135, 154,
          13,  48,  85, 151, 218,  80, 105,  13,  57,  72, 110, 187, 125, 236,
         104, 255, 188, 230, 113,  36,  61, 168, 184, 224,  26, 129, 253, 178,
         254, 182, 114, 141, 117, 219, 140,  16,   6, 184,  86, 181]],
       dtype=torch.uint8)

In [11]:
import numpy as np

# Pack 8 binary values into 1 uint8 byte
def pack_binary(binary_array):
    # Pad to multiple of 8 if necessary
    padded_length = ((len(binary_array) + 7) // 8) * 8
    padded = np.pad(binary_array, (0, padded_length - len(binary_array)))
    
    # Reshape to groups of 8 and pack
    reshaped = padded.reshape(-1, 8)
    packed = np.packbits(reshaped, axis=1).flatten()
    return packed

# Convert -1,1 to 0,1 and pack
x_binary = (x + 1) // 2
x_binary = x_binary.astype(np.uint8)  # Convert to integer type
x_packed = pack_binary(x_binary)

print(f"Original shape: {x.shape}")
print(f"Packed shape: {x_packed.shape}")
print(f"Storage reduction: {x.nbytes / x_packed.nbytes:.1f}x")

Original shape: (1, 768)
Packed shape: (775,)
Storage reduction: 4.0x


In [ ]:
x_packed

tensor([[217, 177,  17,  75,  63,  73,  65, 203, 213,  37, 117, 145, 165, 130,
         157, 109,  39, 177,  98, 175, 174, 188, 216, 197,  90,  64, 229, 114,
          84,  62,  97,  11, 235,  83,  85, 130, 127, 129, 109, 238, 117,  76,
          89, 120,  81,  48, 244,  71,  72, 190, 121, 212,   3, 235, 135, 154,
          13,  48,  85, 151, 218,  80, 105,  13,  57,  72, 110, 187, 125, 236,
         104, 255, 188, 230, 113,  36,  61, 168, 184, 224,  26, 129, 253, 178,
         254, 182, 114, 141, 117, 219, 140,  16,   6, 184,  86, 181]],

array([217, 177,  17,  75,  63,  73,  65, 203, 213,  37, 117, 145, 165,
       130, 157, 109,  39, 177,  98, 175, 174, 188, 216, 197,  90,  64,
       229, 114,  84,  62,  97,  11, 235,  83,  85, 130, 127, 129, 109,
       238, 117,  76,  89, 120,  81,  48, 244,  71,  72, 190, 121, 212,
         3, 235, 135, 154,  13,  48,  85, 151, 218,  80, 105,  13,  57,
        72, 110, 187, 125, 236, 104, 255, 188, 230, 113,  36,  61, 168,
       184, 224,  26, 129, 253, 178, 254, 182, 114, 141, 117, 219, 140,
        16,   6, 184,  86, 181,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   

In [5]:
from datasets import load_dataset
import os
from dotenv import load_dotenv
load_dotenv()
ds = load_dataset("nasa-impact/nasa_repo_code_benchmark_v0.1", token=os.environ["HUGGINGFACE_TOKEN"],)
print(ds)  # See what splits/configs are actually available

Generating train split:   0%|          | 0/239440 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['query-id', 'corpus-id', 'score'],
        num_rows: 239440
    })
})


In [17]:
import sys
from torch import Tensor
from sentence_transformers import SentenceTransformer

# 1. Add the absolute path to the directory containing 'utils.py'
sys.path.append("/rhome/sawale/indus_traning/sentense_transformers/model_exploration/")

# 2. ✅ CRITICAL: Explicitly import your custom class definition
from utils import BinarizationLayer


class UBinarySentenceTransformer(SentenceTransformer):
    """
    A SentenceTransformer model that always outputs binary embeddings.
    """
    def encode(self, sentences, *args, **kwargs) -> Tensor:
        kwargs["precision"] = "ubinary"
        kwargs["convert_to_tensor"] = True
        return super().encode(sentences, *args, **kwargs)


mpath = "/rhome/sawale/indus_traning/sentense_transformers/eval/artifacts/s2_binarized_model/swift-morning-78/checkpoint-1458"

print("--- Loading with UBinarySentenceTransformer ---")
# This should now work correctly and show all layers
model = UBinarySentenceTransformer(
    mpath,
)
print(model)

print("\n--- Loading with standard SentenceTransformer (for comparison) ---")
model2 = SentenceTransformer(
    mpath,
)
print(model2)

No sentence-transformers model found with name /rhome/sawale/indus_traning/sentense_transformers/eval/artifacts/s2_binarized_model/swift-morning-78/checkpoint-1458. Creating a new one with mean pooling.


--- Loading with UBinarySentenceTransformer ---
UBinarySentenceTransformer(
  (0): Transformer({'max_seq_length': 1024, 'do_lower_case': False, 'architecture': 'RobertaModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

--- Loading with standard SentenceTransformer (for comparison) ---
SentenceTransformer(
  (0): Transformer({'max_seq_length': 1024, 'do_lower_case': False, 'architecture': 'RobertaModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Dense({'in_features': 768, 'out_fe

In [19]:
import sys
from torch import Tensor
from sentence_transformers import SentenceTransformer, models

# 1. Add the path and import your custom class as before
sys.path.append("/rhome/sawale/indus_traning/sentense_transformers/model_exploration/")
from utils import BinarizationLayer

# 2. Define your custom class, but we will NOT use it to load the model from a path
class UBinarySentenceTransformer(SentenceTransformer):
    """
    A SentenceTransformer model that always outputs binary embeddings.
    This class is initialized with existing modules, not from a path.
    """
    def encode(self, sentences, *args, **kwargs) -> Tensor:
        """
        Overrides the default encode method to enforce binary precision.
        """
        kwargs["precision"] = "ubinary"
        kwargs["convert_to_tensor"] = True
        return super().encode(sentences, *args, **kwargs)

# The path to your model
mpath = "/rhome/sawale/indus_traning/sentense_transformers/eval/artifacts/s2_binarized_model/swift-morning-78/checkpoint-1458"


# --- The Two-Step Solution ---

# STEP 1: Load the model using the base class, which works reliably.
print("--- Step 1: Loading with base SentenceTransformer ---")
base_model = SentenceTransformer(mpath)
print("Base model loaded successfully:")
print(base_model)


# STEP 2: Create an instance of your custom class using the modules from the loaded model.
print("\n--- Step 2: Wrapping modules with UBinarySentenceTransformer ---")
model = UBinarySentenceTransformer(modules=base_model._modules.values())
print("Custom class instantiated successfully:")
print(model)

--- Step 1: Loading with base SentenceTransformer ---
Base model loaded successfully:
SentenceTransformer(
  (0): Transformer({'max_seq_length': 1024, 'do_lower_case': False, 'architecture': 'RobertaModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Dense({'in_features': 768, 'out_features': 768, 'bias': True, 'activation_function': 'torch.nn.modules.activation.ReLU'})
  (3): BinarizationLayer()
)

--- Step 2: Wrapping modules with UBinarySentenceTransformer ---
Custom class instantiated successfully:
UBinarySentenceTransformer(
  (0): Transformer({'max_seq_length': 1024, 'do_lower_case': False, 'architecture': 'RobertaModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tok

In [20]:
x = base_model.encode(["abc"])
y = model.encode(["abc"])

In [21]:
x

array([[1., 0., 0., 0., 1., 1., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0.,
        1., 1., 1., 0., 1., 0., 0., 0., 1., 1., 1., 0., 0., 0., 1., 1.,
        0., 0., 1., 1., 0., 1., 0., 1., 0., 0., 1., 0., 1., 1., 0., 1.,
        0., 1., 1., 1., 1., 0., 0., 1., 0., 1., 1., 1., 0., 0., 0., 0.,
        0., 1., 0., 0., 0., 1., 0., 1., 1., 0., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 0., 1., 0., 0., 1., 0., 0., 1., 1., 0., 1., 0., 1.,
        0., 1., 1., 0., 1., 0., 0., 1., 0., 1., 1., 1., 0., 1., 0., 1.,
        0., 0., 0., 1., 0., 0., 1., 0., 1., 1., 0., 1., 1., 0., 0., 0.,
        0., 1., 1., 1., 1., 0., 1., 0., 1., 1., 1., 1., 0., 0., 0., 1.,
        1., 1., 1., 0., 0., 1., 1., 0., 1., 1., 1., 1., 0., 1., 1., 0.,
        1., 0., 1., 0., 0., 0., 0., 0., 1., 0., 1., 1., 0., 0., 1., 0.,
        0., 1., 1., 1., 0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 1., 1.,
        0., 0., 1., 1., 0., 1., 0., 1., 1., 0., 1., 0., 0., 1., 1., 1.,
        0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 1., 0., 0., 1., 

In [22]:
y

tensor([[141,   4, 232, 227,  53,  45, 121, 112,  69, 191, 233,  53, 105, 117,
          18, 216, 122, 241, 230, 246, 160, 178, 112,  43,  53, 167,   6,  39,
         251, 177,  23, 227,  92, 129,  14, 251,  31,  91, 180,  93, 222, 230,
         154, 230, 200,  82,  67, 238,  53,  66, 250, 174,  47, 235, 162, 235,
           8, 153, 109, 245, 168,  19, 231, 223,  38, 197, 193, 254, 209, 241,
          83, 234,  64,  72,   3, 229, 136, 218,  13, 158,  81, 222,  36, 217,
         189, 132, 208, 119, 123,  81, 226, 249, 193,  96, 146, 201]],
       dtype=torch.uint8)